# Chatbot Paraphrase Training + Telegram

1. Download & clean the dataset (`generated-par.full.id`)
2. Fine-tune IndoBART (**automatically skipped if a checkpoint already exists in Drive**, making subsequent runs faster)
3. Evaluation (BLEU/ROUGE + length ratio)
4. Paraphrasing function
5. Run the Telegram bot

## Part A Setup & Data

In [ ]:
!pip install gdown pandas transformers sentencepiece huggingface_hub pyTelegramBotAPI sacrebleu rouge-score --quiet
!pip install --no-deps indobenchmark-toolkit --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/skripsi-parafrase"
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/ckpt/best", exist_ok=True)  # will check the contents in Part B

Mounted at /content/drive


In [ ]:
import os
os.makedirs('raw', exist_ok=True)

if not os.path.exists('raw/generated-par.full.id'):
    !gdown "https://drive.google.com/uc?id=1QPyD4lOKxbXGUypA5ke6Y9_i9utq-QSQ" -O raw/generated-par.full.id
else:
    print("Dataset mentah sudah ada, skip download.")

Downloading...
From (original): https://drive.google.com/uc?id=1QPyD4lOKxbXGUypA5ke6Y9_i9utq-QSQ
From (redirected): https://drive.google.com/uc?id=1QPyD4lOKxbXGUypA5ke6Y9_i9utq-QSQ&confirm=t&uuid=45756630-ee82-4381-956b-b11a0c7311c3
To: /content/raw/generated-par.full.id
100% 889M/889M [00:12<00:00, 71.3MB/s]


In [ ]:
import pandas as pd

df = pd.read_csv('raw/generated-par.full.id',
                 sep='\t', header=None, quoting=3,
                 nrows=300_000, dtype=str,
                 on_bad_lines='skip')
df.columns = ['bleu', 'source', 'target']
df['bleu'] = pd.to_numeric(df['bleu'], errors='coerce')
df = df.dropna()
print('Bentuk tabel:', df.shape)
df.head()

Bentuk tabel: (300000, 3)


,bleu,source,target
0,46.825688,Burton tidak pernah bersabar kecuali ketika it...,Burton tidak pernah sabar kecuali bila itu ben...
1,20.556681,Anna tahu kalau Kolom Kelima semakin kuat.,Anna tahu bahwa kelima kolom semakin kuat.
2,37.817904,"Hanya saja, dia benar-benar meminta kita ... t...","Hanya saja, ia benar-benar meminta kami ... ti..."
3,25.394297,Tak ada yang seperti meninggalkan rumah sakit ...,Tidak ada yang cukup mirip meninggalkan rumah ...
4,30.213754,Leibniz berhenti dan mempertimbangkan masalah ...,Leibniz berhenti dan memikirkan hal-hal.


In [ ]:
def word_set(s):
    return set(str(s).lower().split())

before = len(df)

# Similarity score filter
df = df[df['bleu'].between(10, 40)]
print(f'Setelah filter skor 10-40 : {len(df):,} (dari {before:,})')

# Discard the remaining English
eng = r'\b(the|and|is|are|was|were|have|has|had|that|this|with|you|your|for)\b'
mask_eng = df['target'].str.lower().str.contains(eng, regex=True)
df = df[~mask_eng]
print(f'Setelah buang sisa bahasa Inggris : {len(df):,}')

# Discard identical pairs
df = df[df['source'].str.lower() != df['target'].str.lower()]
print(f'Setelah buang pasangan identik : {len(df):,}')

# Root word length filter
st = df['source'].str.split().str.len()
tt = df['target'].str.split().str.len()
df = df[st.between(3, 40) & tt.between(3, 40)]
print(f'Setelah filter panjang 3-40 kata : {len(df):,}')

# 1) length ratio tightened (0.8–1.3, instead of 0.5–2.0)
st = df['source'].str.split().str.len()
tt = df['target'].str.split().str.len()
rasio = tt / st
df = df[rasio.between(0.8, 1.3)]
print(f'Setelah filter rasio panjang ketat (0.8-1.3) : {len(df):,}')

# 2) Discard targets whose vocabulary is a pure subset of the source (a characteristic of extractive compression).
is_subset = df.apply(lambda r: word_set(r['target']).issubset(word_set(r['source'])), axis=1)
df = df[~is_subset]
print(f'Setelah buang target subset kosakata source : {len(df):,}')

# 3) novel word ratio -> the target must contain a sufficient number of new words (an indication of genuine lexical substitution)
def novel_word_ratio(row):
    tgt = word_set(row['target'])
    if not tgt:
        return 0.0
    return len(tgt - word_set(row['source'])) / len(tgt)

df['novel_word_ratio'] = df.apply(novel_word_ratio, axis=1)
df = df[df['novel_word_ratio'] > 0.15]
df = df.drop(columns=['novel_word_ratio'])
print(f'Setelah filter novel_word_ratio > 0.15 : {len(df):,}')

df = df.drop_duplicates(['source', 'target'])
df = df.drop_duplicates('source')
df = df.sample(n=min(30_000, len(df)), random_state=42).reset_index(drop=True)
print(f'Sampel akhir : {len(df):,}')

Setelah filter skor 10-40 : 199,915 (dari 300,000)


/tmp/ipykernel_1660/2928108967.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_eng = df['target'].str.lower().str.contains(eng, regex=True)


Setelah buang sisa bahasa Inggris : 198,873
Setelah buang pasangan identik : 198,873
Setelah filter panjang 3-40 kata : 197,497
Setelah filter rasio panjang ketat (0.8-1.3) : 149,751
Setelah buang target subset kosakata source : 149,505
Setelah filter novel_word_ratio > 0.15 : 146,469
Sampel akhir : 30,000


In [ ]:
test = df.iloc[:2000]
val = df.iloc[2000:4000]
train = df.iloc[4000:]

os.makedirs('data', exist_ok=True)
for nama, bagian in [('train', train), ('val', val), ('test', test)]:
    bagian[['source', 'target']].to_csv(f'data/{nama}.csv', index=False)
    print(f'{nama:6} : {len(bagian):>6,} baris')

!cp -r data {DRIVE_DIR}/
print('Data disimpan ke Drive.')

train  : 26,000 baris
val    :  2,000 baris
test   :  2,000 baris
Data disimpan ke Drive.


## Part B: Model & Tokenizer

First, check whether a checkpoint from a previous training run already exists in your Drive. If it does, use it (and skip the training step in Part C). If not, you will start from the base model `indobenchmark/indobart-v2`.

In [ ]:
import torch
from transformers import MBartForConditionalGeneration
from huggingface_hub import hf_hub_download
import sentencepiece as spm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_DRIVE = f"{DRIVE_DIR}/ckpt/best"
SUDAH_ADA_CHECKPOINT = os.path.exists(f"{CKPT_DRIVE}/config.json")

sp_path = hf_hub_download('indobenchmark/indobart-v2', 'sentencepiece.bpe.model')
sp = spm.SentencePieceProcessor()
sp.load(sp_path)

if SUDAH_ADA_CHECKPOINT:
    print("Checkpoint ditemukan di Drive -> load model hasil training sebelumnya, skip training.")
    model = MBartForConditionalGeneration.from_pretrained(CKPT_DRIVE)
else:
    print("Belum ada checkpoint -> mulai dari model dasar indobart-v2, akan training di Bagian C.")
    model = MBartForConditionalGeneration.from_pretrained('indobenchmark/indobart-v2')

model.to(device)
print('Model siap di:', device)

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B /  932kB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

Belum ada checkpoint -> mulai dari model dasar indobart-v2, akan training di Bagian C.


Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

Model siap di: cuda


In [ ]:
OFFSET = 4
BOS, PAD, EOS, UNK = 0, 1, 2, 3

def encode(teks, max_len=64):
    ids = [i + OFFSET for i in sp.encode_as_ids(teks)][:max_len - 1] + [EOS]
    return ids

def decode(ids):
    bersih = [i - OFFSET for i in ids if i >= OFFSET]
    return sp.decode_ids(bersih)

# Back-and-forth test
teks = "Penelitian ini menggunakan pendekatan kuantitatif."
print('Ids    :', encode(teks))
print('Decode :', decode(encode(teks)))

Ids    : [39939, 88, 266, 335, 1763, 365, 778, 4925, 18580, 39959, 2]
Decode : Penelitian ini menggunakan pendekatan kuantitatif.


## Part C: Training (automatically skipped if a checkpoint already exists)

This cell only performs actual training if `SUDAH_ADA_CHECKPOINT` is set to `False`. If training has already been conducted previously, this cell will be skipped in a matter of seconds.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import time

class ParafraseDataset(Dataset):
    def __init__(self, df, max_len=64):
        self.df, self.max_len = df.reset_index(drop=True), max_len
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return encode(r.source, self.max_len), encode(r.target, self.max_len)

def collate(batch):
    def pad(seqs):
        m = max(len(s) for s in seqs)
        return torch.tensor([s + [PAD] * (m - len(s)) for s in seqs])
    src, tgt = zip(*batch)
    src, tgt = pad(src), pad(tgt)
    labels = tgt.clone()
    labels[labels == PAD] = -100
    return {'input_ids': src,
            'attention_mask': (src != PAD).long(),
            'labels': labels}

In [ ]:
if not SUDAH_ADA_CHECKPOINT:
    EPOCHS, BATCH, LR = 3, 16, 3e-5

    train_df = pd.read_csv('data/train.csv')
    val_df   = pd.read_csv('data/val.csv')

    train_dl = DataLoader(ParafraseDataset(train_df), batch_size=BATCH,
                          shuffle=True, collate_fn=collate)
    val_dl   = DataLoader(ParafraseDataset(val_df), batch_size=BATCH,
                          shuffle=False, collate_fn=collate)

    opt = AdamW(model.parameters(), lr=LR)
    total = len(train_dl) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)

    os.makedirs('ckpt', exist_ok=True)
    best = float('inf')

    for ep in range(1, EPOCHS + 1):
        model.train()
        t0, jml = time.time(), 0.0
        for step, b in enumerate(train_dl):
            b = {k: v.to(device) for k, v in b.items()}
            loss = model(**b).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            jml += loss.item()
            if step % 200 == 0:
                print(f'  ep{ep} step {step:4d}/{len(train_dl)} | loss {loss.item():.4f}')

        model.eval()
        vjml = 0.0
        with torch.no_grad():
            for b in val_dl:
                b = {k: v.to(device) for k, v in b.items()}
                vjml += model(**b).loss.item()

        tr, va = jml / len(train_dl), vjml / len(val_dl)
        print(f'EPOCH {ep} | train {tr:.4f} | val {va:.4f} | {(time.time()-t0)/60:.1f} menit')

        if va < best:
            best = va
            model.save_pretrained('ckpt/best')
            print(f'  checkpoint disimpan (val {va:.4f})')

    !cp -r ckpt/best {CKPT_DRIVE}
    print('Training selesai & checkpoint disalin ke Drive. Val terbaik:', round(best, 4))
else:
    print('Skip training (checkpoint sudah ada).')

  ep1 step    0/1625 | loss 10.2378
  ep1 step  200/1625 | loss 3.6035
  ep1 step  400/1625 | loss 2.9922
  ep1 step  600/1625 | loss 2.9954
  ep1 step  800/1625 | loss 2.4395
  ep1 step 1000/1625 | loss 2.2791
  ep1 step 1200/1625 | loss 1.7751
  ep1 step 1400/1625 | loss 2.6614
  ep1 step 1600/1625 | loss 1.8685
EPOCH 1 | train 2.9039 | val 2.0735 | 5.0 menit


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  checkpoint disimpan (val 2.0735)
  ep2 step    0/1625 | loss 1.6893
  ep2 step  200/1625 | loss 1.8467
  ep2 step  400/1625 | loss 1.8235
  ep2 step  600/1625 | loss 1.7713
  ep2 step  800/1625 | loss 1.6341
  ep2 step 1000/1625 | loss 1.7843
  ep2 step 1200/1625 | loss 1.9712
  ep2 step 1400/1625 | loss 1.8801
  ep2 step 1600/1625 | loss 1.9491
EPOCH 2 | train 1.8720 | val 1.8926 | 5.1 menit


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  checkpoint disimpan (val 1.8926)
  ep3 step    0/1625 | loss 1.7010
  ep3 step  200/1625 | loss 1.6796
  ep3 step  400/1625 | loss 1.6898
  ep3 step  600/1625 | loss 1.3116
  ep3 step  800/1625 | loss 1.7273
  ep3 step 1000/1625 | loss 1.6798
  ep3 step 1200/1625 | loss 1.1705
  ep3 step 1400/1625 | loss 1.9959
  ep3 step 1600/1625 | loss 1.5484
EPOCH 3 | train 1.6250 | val 1.8442 | 5.0 menit


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  checkpoint disimpan (val 1.8442)
Training selesai & checkpoint disalin ke Drive. Val terbaik: 1.8442


## Part D Evaluation

Check the quality of the results: use BLEU/ROUGE as usual, **plus the length ratio** (output/input) as a direct diagnostic for the "over-shortening" issue we have encountered previously.

In [ ]:
import sacrebleu
from rouge_score import rouge_scorer
import numpy as np
from tqdm import tqdm

test_df = pd.read_csv('data/test.csv')
model.eval()

def generate_batch(teks_list, batch=32):
    hasil = []
    for i in tqdm(range(0, len(teks_list), batch)):
        chunk = teks_list[i:i+batch]
        ids = [encode(t) for t in chunk]
        m = max(len(x) for x in ids)
        inp = torch.tensor([x + [PAD]*(m-len(x)) for x in ids]).to(device)
        mask = (inp != PAD).long()
        min_len = max(5, int(np.mean([len(x) for x in ids]) * 0.8))
        with torch.no_grad():
            out = model.generate(inp, attention_mask=mask, max_length=64, min_length=min_len,
                                 num_beams=5, length_penalty=1.6, early_stopping=True)
        hasil += [decode(o.tolist()) for o in out]
    return hasil

sumber    = test_df['source'].tolist()
referensi = test_df['target'].tolist()
prediksi  = generate_batch(sumber)

bleu = sacrebleu.corpus_bleu(prediksi, [referensi]).score
copy_bleu = sacrebleu.corpus_bleu(prediksi, [sumber]).score

rs = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
sk = [rs.score(r, p) for r, p in zip(referensi, prediksi)]

len_ratio = np.mean([len(p.split()) / max(len(s.split()), 1) for s, p in zip(sumber, prediksi)])

print(f'BLEU (vs referensi) : {bleu:.2f}')
print(f'BLEU (vs input)     : {copy_bleu:.2f}   <- makin tinggi = makin menyalin/menyingkat')
for k in ['rouge1', 'rouge2', 'rougeL']:
    print(f'{k:8} : {np.mean([s[k].fmeasure for s in sk])*100:.2f}')
print(f'Rasio panjang (output/input) : {len_ratio:.2f}   <- makin dekat 1.0 makin baik')

100%|██████████| 63/63 [01:00<00:00,  1.03it/s]


BLEU (vs referensi) : 31.70
BLEU (vs input)     : 49.88   <- makin tinggi = makin menyalin/menyingkat
rouge1   : 59.77
rouge2   : 32.35
rougeL   : 58.46
Rasio panjang (output/input) : 1.06   <- makin dekat 1.0 makin baik


## Part E Paraphrase Function (used by bots)

In [ ]:
def chatbot_paraphrase(teks, length_penalty=1.6):
    if not teks or not teks.strip():
        return "Kirim satu kalimat ya, nanti saya paraphrase-kan."

    ids = encode(teks)
    inp = torch.tensor([ids]).to(device)
    min_len = max(5, int(len(ids) * 0.8))

    with torch.no_grad():
        out = model.generate(
            inp, max_length=64, min_length=min_len,
            num_beams=5, no_repeat_ngram_size=3,
            length_penalty=length_penalty, early_stopping=True
        )
    hasil = decode(out[0].tolist())
    return hasil if hasil else "(maaf, saya belum bisa menghasilkan paraphrase untuk kalimat ini)"

print(chatbot_paraphrase("Saya ingin belajar pemrograman dengan cepat."))

Aku mau belajar dari pemrograman dengan cepat.


## Part F Run the Telegram Bot

In [ ]:
try:
    from google.colab import userdata
    TELEGRAM_TOKEN = userdata.get('TELEGRAM_TOKEN')
except Exception:
    TELEGRAM_TOKEN = None

if not TELEGRAM_TOKEN:
    TELEGRAM_TOKEN = input("Masukkan Telegram bot token (dari @BotFather): ").strip()

import telebot

bot = telebot.TeleBot(TELEGRAM_TOKEN)

@bot.message_handler(commands=['start', 'help'])
def send_welcome(message):
    welcome_text = "👋 Selamat datang di Chatbot Parafrase!\n\nKirimkan kalimat apa saja, dan saya akan memparafrasekannya secara otomatis."
    bot.reply_to(message, welcome_text)

@bot.message_handler(func=lambda msg: True)
def handle_paraphrase_request(message):
    incoming_text = message.text.strip()
    print(f"[Telegram] User: {incoming_text}")
    try:
        reply_text = chatbot_paraphrase(incoming_text)
    except Exception as e:
        print("Error saat generate:", e)
        reply_text = "Maaf, ada masalah saat memproses kalimat ini."
    print(f"[Telegram] Bot: {reply_text}")
    bot.reply_to(message, reply_text)

print("🚀 Bot Telegram aktif! Silakan chat dengan bot Anda.")
bot.infinity_polling()